<a href="https://colab.research.google.com/github/zeeofficial01/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zeeofficial01/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 — CTR vs. average search position: CONFIRMED**

CTR is generally higher for content with better average search positions and lower for content with worse positions. This supports using CTR relative to position as a baseline review signal.

**Rule:** Rank each content item using two signals:

1. CTR compared with its average search position
2. Search traffic volume (GSC impressions)

A higher score means the item should be reviewed first.

**Signal check 2 — Search traffic volume: CONFIRMED**

Search traffic volume is strongly separated across the four buckets, from a median of 4 impressions in the Low bucket to 3,012 in the Very High bucket. This supports using GSC impressions as a baseline prioritization signal.

**Final baseline rule:** Give each content item a score based on:
1. CTR compared with its average search position
2. Search traffic volume (GSC impressions)

A higher score means the item should be reviewed first.

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

columns_needed = [
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions"
]

march = pd.read_parquet(
    file_path,
    columns=columns_needed
)

print("Rows loaded:", len(march))
print("Columns:", march.columns.tolist())

display(march.head())

Rows loaded: 9841378
Columns: ['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN


In [ ]:
# Create monthly content-level data for the baseline checks

march["gsc_impressions"] = pd.to_numeric(
    march["gsc_impressions"], errors="coerce"
).fillna(0)

march["gsc_clicks"] = pd.to_numeric(
    march["gsc_clicks"], errors="coerce"
).fillna(0)

march["gsc_avg_position"] = pd.to_numeric(
    march["gsc_avg_position"], errors="coerce"
)

march["ga4_sessions"] = pd.to_numeric(
    march["ga4_sessions"], errors="coerce"
).fillna(0)

content = (
    march.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_sessions=("ga4_sessions", "sum")
    )
)

content["ctr"] = np.where(
    content["gsc_impressions"] > 0,
    content["gsc_clicks"] / content["gsc_impressions"],
    np.nan
)

print("Content items:", len(content))
display(content.head())

Content items: 331437


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,NaN


In [ ]:
# Signal Check 1: CTR vs. Average Search Position

check = content[
    (content["gsc_impressions"] > 0) &
    (content["gsc_avg_position"].notna()) &
    (content["ctr"].notna())
].copy()

check["position_bucket"] = pd.cut(
    check["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_check = (
    check.groupby("position_bucket", observed=False)
    .agg(
        median_ctr=("ctr", "median"),
        n=("ctr", "size")
    )
    .reset_index()
)

print("Signal: CTR vs. average search position")
display(position_check)


Signal: CTR vs. average search position


,position_bucket,median_ctr,n
0,1-3,0.0,17578
1,4-10,0.0,81988
2,11-20,0.0,32203
3,21+,0.0,44969


In [ ]:
# Inspect CTR vs. position among items with at least one click

clicked_check = check[check["gsc_clicks"] > 0].copy()

clicked_position_check = (
    clicked_check.groupby("position_bucket", observed=False)
    .agg(
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        n=("ctr", "size")
    )
    .reset_index()
)

print("CTR vs. position among items with at least one click")
display(clicked_position_check)

CTR vs. position among items with at least one click


,position_bucket,median_ctr,mean_ctr,n
0,1-3,0.003507,0.027709,7866
1,4-10,0.003398,0.010939,36922
2,11-20,0.003178,0.007895,13098
3,21+,0.002117,0.007916,10951


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Signal Check 2: Search traffic volume

volume_check = content[
    content["gsc_impressions"] > 0
].copy()

volume_check["volume_bucket"] = pd.qcut(
    volume_check["gsc_impressions"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

volume_table = (
    volume_check.groupby("volume_bucket", observed=False)
    .agg(
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        n=("content_hash_id", "size")
    )
    .reset_index()
)

print("Signal: Search traffic volume (GSC impressions)")
display(volume_table)

Signal: Search traffic volume (GSC impressions)


,volume_bucket,median_impressions,median_ctr,n
0,Low,4.0,0.000000,44983
1,Medium,67.0,0.000000,43409
2,High,419.0,0.000000,44186
3,Very High,3012.0,0.001943,44160


In [ ]:
# Section 2: Prepare the scoring data

scored = content.copy()

scored["position_bucket"] = pd.cut(
    scored["gsc_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

print("Rows ready for scoring:", len(scored))
display(scored.head())

Rows ready for scoring: 331437


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ctr,position_bucket
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0,4-10
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,NaN,NaN


In [ ]:
# Calculate the expected CTR for each position bucket

position_reference = (
    check.groupby("position_bucket", observed=False)
    .agg(
        reference_ctr=("ctr", "mean")
    )
    .reset_index()
)

scored = scored.merge(
    position_reference,
    on="position_bucket",
    how="left"
)

print("Scoring data prepared.")
display(scored.head())

Scoring data prepared.


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ctr,position_bucket,reference_ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,NaN,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,NaN,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,NaN,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0,4-10,0.004926
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,NaN,NaN,NaN


In [ ]:
# Calculate how far each item's CTR is below its position-based reference

scored["ctr_gap"] = (
    scored["reference_ctr"] - scored["ctr"]
)

print("CTR gap calculated.")
display(
    scored[
        [
            "content_hash_id",
            "gsc_impressions",
            "ctr",
            "position_bucket",
            "reference_ctr",
            "ctr_gap"
        ]
    ].head(10)
)

CTR gap calculated.


,content_hash_id,gsc_impressions,ctr,position_bucket,reference_ctr,ctr_gap
0,content_004e9c4c32e88631,0,NaN,NaN,NaN,NaN
1,content_0236ef736698e17c,0,NaN,NaN,NaN,NaN
2,content_025f6cfd3c298870,0,NaN,NaN,NaN,NaN
3,content_0263d5f9b7a2ecd4,1,0.000000,4-10,0.004926,0.004926
4,content_02752c6c1c60161f,0,NaN,NaN,NaN,NaN
5,content_0317b24cc1ff5c5d,0,NaN,NaN,NaN,NaN
6,content_044c54ec4adcc4b2,0,NaN,NaN,NaN,NaN
7,content_04c67f3541177192,331,0.006042,11-20,0.003211,-0.002831
8,content_05acc92c165f4386,33,0.000000,4-10,0.004926,0.004926
9,content_07573a1cc2034981,0,NaN,NaN,NaN,NaN


In [ ]:
# Create the two-part baseline score

# 1 point when CTR is below the position-based reference
scored["ctr_signal"] = (
    scored["ctr_gap"] > 0
).astype(int)

# 1 point for high search traffic volume
volume_threshold = scored["gsc_impressions"].quantile(0.75)

scored["volume_signal"] = (
    scored["gsc_impressions"] >= volume_threshold
).astype(int)

# Total baseline score
scored["score"] = (
    scored["ctr_signal"] +
    scored["volume_signal"]
)

print("Volume threshold:", volume_threshold)
print("Score distribution:")
display(scored["score"].value_counts().sort_index())

Volume threshold: 216.0
Score distribution:


,count
score,
0,163815
1,101541
2,66081


In [ ]:
# Assign one reason code and one action label

scored["reason_code"] = np.select(
    [
        (scored["ctr_signal"] == 1) & (scored["volume_signal"] == 1),
        (scored["ctr_signal"] == 1),
        (scored["volume_signal"] == 1)
    ],
    [
        "CTR_AND_VOLUME",
        "CTR_BELOW_POSITION_REFERENCE",
        "HIGH_SEARCH_VOLUME"
    ],
    default="NO_SIGNAL"
)

scored["action"] = np.select(
    [
        scored["score"] == 2,
        scored["score"] == 1
    ],
    [
        "PRIORITY_REVIEW",
        "REVIEW"
    ],
    default="NO_ACTION"
)

print("Reason codes:")
display(scored["reason_code"].value_counts())

print("Actions:")
display(scored["action"].value_counts())

Reason codes:


,count
reason_code,
NO_SIGNAL,163815
CTR_BELOW_POSITION_REFERENCE,84749
CTR_AND_VOLUME,66081
HIGH_SEARCH_VOLUME,16792


Actions:


,count
action,
NO_ACTION,163815
REVIEW,101541
PRIORITY_REVIEW,66081


In [ ]:
# Rank the queue and prepare the baseline output

scored = scored.sort_values(
    by=["score", "gsc_impressions", "ctr_gap"],
    ascending=[False, False, False]
).reset_index(drop=True)

scored["rank"] = scored.index + 1

baseline_queue = scored[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "ctr_gap"
    ]
].copy()

print("Ranked queue rows:", len(baseline_queue))
display(baseline_queue.head(10))

Ranked queue rows: 331437


,rank,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ctr_gap
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,2,CTR_AND_VOLUME,PRIORITY_REVIEW,617124,5668,0.009185,2.383011,0.003215
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,2,CTR_AND_VOLUME,PRIORITY_REVIEW,245276,1480,0.006034,2.854514,0.006365
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,2,CTR_AND_VOLUME,PRIORITY_REVIEW,244931,669,0.002731,15.008339,0.000480
3,4,client_e547b89c05043229,content_0e03de7680314cd5,2,CTR_AND_VOLUME,PRIORITY_REVIEW,221310,720,0.003253,2.675217,0.009146
4,5,client_23a62021009f63c4,content_44f34c0a90047651,2,CTR_AND_VOLUME,PRIORITY_REVIEW,212404,24,0.000113,7.346909,0.004813
5,6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,2,CTR_AND_VOLUME,PRIORITY_REVIEW,205867,862,0.004187,3.367835,0.000739
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,2,CTR_AND_VOLUME,PRIORITY_REVIEW,203497,289,0.001420,2.563756,0.010979
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,2,CTR_AND_VOLUME,PRIORITY_REVIEW,194579,242,0.001244,32.766674,0.000684
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,2,CTR_AND_VOLUME,PRIORITY_REVIEW,194337,361,0.001858,4.450106,0.003068
9,10,client_e547b89c05043229,content_4ffe18112a5642e3,2,CTR_AND_VOLUME,PRIORITY_REVIEW,186983,586,0.003134,2.331060,0.009265


In [ ]:
# Write the ranked baseline queue to the required CSV file

import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print("CSV written successfully:")
print(output_path)
print("Rows:", len(baseline_queue))

CSV written successfully:
work/outputs/baseline_action_score.csv
Rows: 331437


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Section 3: Show the Top-20 for review

top20 = baseline_queue.head(20).copy()

display(
    top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "ctr_gap"
        ]
    ]
)

,rank,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ctr_gap
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,2,CTR_AND_VOLUME,PRIORITY_REVIEW,617124,5668,0.009185,2.383011,0.003215
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,2,CTR_AND_VOLUME,PRIORITY_REVIEW,245276,1480,0.006034,2.854514,0.006365
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,2,CTR_AND_VOLUME,PRIORITY_REVIEW,244931,669,0.002731,15.008339,0.000480
3,4,client_e547b89c05043229,content_0e03de7680314cd5,2,CTR_AND_VOLUME,PRIORITY_REVIEW,221310,720,0.003253,2.675217,0.009146
4,5,client_23a62021009f63c4,content_44f34c0a90047651,2,CTR_AND_VOLUME,PRIORITY_REVIEW,212404,24,0.000113,7.346909,0.004813
5,6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,2,CTR_AND_VOLUME,PRIORITY_REVIEW,205867,862,0.004187,3.367835,0.000739
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,2,CTR_AND_VOLUME,PRIORITY_REVIEW,203497,289,0.001420,2.563756,0.010979
7,8,client_23a62021009f63c4,content_36e53e9c707674fc,2,CTR_AND_VOLUME,PRIORITY_REVIEW,194579,242,0.001244,32.766674,0.000684
8,9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,2,CTR_AND_VOLUME,PRIORITY_REVIEW,194337,361,0.001858,4.450106,0.003068
9,10,client_e547b89c05043229,content_4ffe18112a5642e3,2,CTR_AND_VOLUME,PRIORITY_REVIEW,186983,586,0.003134,2.331060,0.009265


### Top-20 review

| Rank | Action | Why it is here | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Strong signal because both conditions are met and impressions are very high. | The CTR gap may not represent a meaningful opportunity after considering query mix or intent. |
| 2 | PRIORITY_REVIEW | Very high search volume with CTR below the position reference. | Strong signal because both baseline conditions are met. | Different query intent or SERP features could explain the lower CTR. |
| 3 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Moderate confidence because the CTR gap is relatively small. | The small CTR gap may be too weak to justify priority despite high volume. |
| 4 | PRIORITY_REVIEW | Very high search volume with a clear CTR gap below the position reference. | Strong signal because the CTR gap is large and volume is high. | The observed CTR difference may be caused by query mix or search-result context. |
| 5 | PRIORITY_REVIEW | Very high search volume and very low observed CTR relative to its position reference. | Strong opportunity signal because the item receives many impressions but few clicks. | The low CTR may be expected for the queries or SERP features involved. |
| 6 | PRIORITY_REVIEW | Very high search volume with CTR below the position reference. | Moderate confidence because the CTR gap is relatively small. | The small gap may not indicate enough improvement potential. |
| 7 | PRIORITY_REVIEW | Very high search volume with a large CTR gap below the position reference. | Strong signal because both volume and CTR-gap conditions are substantial. | Query intent or SERP layout could explain the CTR difference. |
| 8 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Moderate confidence because the CTR gap is small. | The small CTR gap may make this a weaker priority than the score suggests. |
| 9 | PRIORITY_REVIEW | Very high search volume with CTR below the position reference. | Strong enough for review because both baseline signals are present. | The CTR gap may reflect query mix rather than an actionable issue. |
| 10 | PRIORITY_REVIEW | Very high search volume and a clear CTR gap below the position reference. | Strong signal because volume and CTR-gap conditions are both met. | SERP features or query intent could reduce the expected CTR. |
| 11 | PRIORITY_REVIEW | Very high search volume with CTR below the position reference. | Strong enough for baseline review because both signals agree. | The relationship may not hold for the item's specific queries. |
| 12 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Moderate-to-strong confidence because both conditions are met. | Query mix or SERP context could make the lower CTR reasonable. |
| 13 | PRIORITY_REVIEW | Very high search volume with a substantial CTR gap below the position reference. | Strong signal because the item combines high volume with a sizeable gap. | The reference CTR may not be an appropriate comparison for this item's query mix. |
| 14 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Moderate confidence because the CTR gap is smaller than some higher-ranked items. | The gap may be too small to create a meaningful improvement opportunity. |
| 15 | PRIORITY_REVIEW | Very high search volume with CTR below the position reference. | Moderate confidence because position is relatively poor and the CTR gap is modest. | Poor position itself may explain the low CTR, limiting the value of a CTR-focused action. |
| 16 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Strong signal because the CTR gap is meaningful despite the good position. | The CTR difference could come from query intent or SERP features. |
| 17 | PRIORITY_REVIEW | Very high search volume with CTR below the position reference. | Moderate-to-strong confidence because both signals are present. | The observed gap may not translate into an actionable improvement. |
| 18 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Moderate confidence because the CTR gap is relatively small and position is poor. | The poor position may itself explain the low CTR. |
| 19 | PRIORITY_REVIEW | Very high search volume with a clear CTR gap below the position reference. | Strong signal because both volume and CTR-gap conditions are met. | Query mix or SERP context could explain the difference. |
| 20 | PRIORITY_REVIEW | Very high search volume and CTR below the position reference. | Moderate-to-strong confidence because both baseline signals are present. | The CTR gap may not represent a meaningful optimization opportunity. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Section 4: Identify potentially weak picks

weak_picks = top20[
    top20["ctr_gap"] < 0.001
].copy()

print("Potentially weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "gsc_impressions",
            "ctr",
            "gsc_avg_position",
            "ctr_gap"
        ]
    ]
)

Potentially weak picks:


,rank,content_hash_id,score,reason_code,action,gsc_impressions,ctr,gsc_avg_position,ctr_gap
2,3,content_e8a52cf3d5988c07,2,CTR_AND_VOLUME,PRIORITY_REVIEW,244931,0.002731,15.008339,0.000480
5,6,content_7172a7fad43f0998,2,CTR_AND_VOLUME,PRIORITY_REVIEW,205867,0.004187,3.367835,0.000739
7,8,content_36e53e9c707674fc,2,CTR_AND_VOLUME,PRIORITY_REVIEW,194579,0.001244,32.766674,0.000684
17,18,content_3df3f32f3fd58dea,2,CTR_AND_VOLUME,PRIORITY_REVIEW,140156,0.001406,23.335465,0.000522


### Weak picks

Four of the Top-20 look relatively weak under manual review: ranks 3, 6, 8, and 18.

They all receive `PRIORITY_REVIEW` because they have high search volume and their CTR is below the position-based reference. However, their CTR gaps are all below 0.001, so the difference from the reference is small.

This shows a limitation of the baseline rule: the score is binary, so a very small CTR gap receives the same CTR signal as a much larger gap. These picks should therefore be treated as lower-confidence opportunities rather than confirmed optimization targets.

### Leakage check

No future-window or label-derived inputs were used in the scoring rule. The score uses March 2026 aggregated GSC impressions, GSC clicks, CTR, and average search position.

No product flags were used as scoring features. The CTR-vs-position signal was selected because it is linked to the session's CTR-fix logic, but the actual score uses only the observed March performance signals.

## Self-check

- [x] Two signal checks are included with visible bucket tables and `n`.
- [x] At least one signal is linked to a real FlyRank flag.
- [x] One baseline rule produces a score, one reason code, and one action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Top-20 review is included, covering the required Top-10.
- [x] Weak picks are identified and explained.
- [x] No future-window or label-derived inputs are used.
- [x] No product flags are used as scoring features.
- [ ] Notebook runs top-to-bottom with no errors.
- [ ] Notebook is saved/committed under `work/notebooks/w04_baseline_score.ipynb`.